In [2]:
import pandas as pd
import numpy as np

In [19]:
# Extract comprehensive sample annotations from the GEO series matrix file
import re

# Dictionary to store all annotation rows
annotations_dict = {}

with open('../data/GSE240671_series_matrix.txt', 'r') as f:
    for line in f:
        line = line.strip()
        if line.startswith('!Sample_geo_accession'):
            # Extract all GSM IDs
            sample_ids = re.findall(r'"([^"]*)"', line)
        elif line.startswith('!Sample_characteristics_ch1'):
            # Extract the characteristics data
            characteristics = re.findall(r'"([^"]*)"', line)
            if characteristics:
                # Parse the first characteristic to get the annotation type
                first_char = characteristics[0]
                if ':' in first_char:
                    annotation_type = first_char.split(':')[0].strip()
                    
                    # Extract values for each sample
                    values = []
                    for char in characteristics:
                        if ':' in char:
                            value = char.split(':', 1)[1].strip()
                            values.append(value)
                        else:
                            values.append(char)
                    
                    annotations_dict[annotation_type] = values

# Create DataFrame with sample names and all annotations
sample_annotation_df = pd.DataFrame({'sample_name': sample_ids})

# Add all annotation columns
for annotation_type, values in annotations_dict.items():
    if len(values) == len(sample_ids):  # Ensure same length
        sample_annotation_df[annotation_type] = values

# Parse sample titles to add structured columns
def extract_timing_and_patient(sample_name_idx):
    timing_values = annotations_dict.get('timing biopsies', [])
    if sample_name_idx < len(timing_values):
        timing = timing_values[sample_name_idx]
        if 'pre-surgery' in timing:
            return 'pre-treatment'
        elif 'post-surgery' in timing:
            return 'post-treatment'
    return 'unknown'

def extract_patient_number(sample_name_idx):
    patient_values = annotations_dict.get('patient id', [])
    if sample_name_idx < len(patient_values):
        patient_id = patient_values[sample_name_idx]
        # Extract number from patient ID like "R01_UCL" or "P01_UCL"
        import re
        match = re.search(r'([RP])(\d+)', patient_id)
        if match:
            return f"Patient{match.group(2)}"
    return 'unknown'

# Add derived columns
sample_annotation_df['treatment_status'] = [extract_timing_and_patient(i) for i in range(len(sample_ids))]
sample_annotation_df['patient_number'] = [extract_patient_number(i) for i in range(len(sample_ids))]

print(f'Extracted {len(sample_annotation_df)} samples with {len(sample_annotation_df.columns)} annotation columns')
sample_annotation_df

Extracted 122 samples with 24 annotation columns


,sample_name,tissue,timing biopsies,patient id,Sex,age diagnostic,statut menopausal,tum size_max_diagnostic_(larger_diameter_in_mm),molecular category,ki67 binary_(1_>_and_=_15%_et_0_<15%),...,nac category,nac herceptin_binary,breast conservative_surgery,sentinel lymph_node,radiotherapy binary,hormonotherapy type_category,sequencing batch,mapped to_intergenic_(%_reads),treatment_status,patient_number
0,GSM7707262,mammary tumor,pre-surgery,R01_UCL,female,57,yes,21,Luminal B,1,...,anthracyclines and taxanes,no,yes,no,yes,Femara,batch_oct_2019,7.54,pre-treatment,Patient01
1,GSM7707263,mammary tumor,pre-surgery,R02_UCL,female,33,no,100,TNBC,1,...,anthracyclines and taxanes,no,no,no,yes,no,batch_oct_2019,8.13,pre-treatment,Patient02
2,GSM7707264,mammary tumor,pre-surgery,R03_UCL,female,34,no,40,Luminal B,1,...,anthracyclin alone,no,no,no,no,Arimidex,batch_oct_2019,9.08,pre-treatment,Patient03
3,GSM7707265,mammary tumor,pre-surgery,R04_UCL,female,75,yes,8,Luminal A,0,...,anthracyclin alone,no,no,no,yes,Femara,batch_oct_2019,7.41,pre-treatment,Patient04
4,GSM7707266,mammary tumor,pre-surgery,R05_UCL,female,32,no,29,Luminal B,1,...,anthracyclines and taxanes,no,no,no,yes,Femara,batch_oct_2019,7.85,pre-treatment,Patient05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117,GSM7707379,mammary tumor,pre-surgery,P61_UCL,female,56,yes,24,HER2+,1,...,anthracyclines and taxanes,yes,yes,no,yes,no,batch_dec_2022,54.883,pre-treatment,Patient61
118,GSM7707380,mammary tumor,pre-surgery,P62_UCL,female,31,no,40,Luminal B,1,...,anthracyclines and taxanes,no,yes,no,yes,Nolvadex,batch_dec_2022,28.65,pre-treatment,Patient62
119,GSM7707381,mammary tumor,pre-surgery,P64_UCL,female,52,no,11,HER2+_HR+,1,...,anthracyclines and taxanes,yes,yes,yes,yes,Femara,batch_dec_2022,59.892,pre-treatment,Patient64
120,GSM7707382,mammary tumor,pre-surgery,P65_UCL,female,36,no,118,Luminal B,1,...,anthracyclines and taxanes,no,no,no,yes,Femara,batch_dec_2022,28.326,pre-treatment,Patient65


In [22]:
# Check if patients appear in both pre-treatment and post-treatment groups
pre_treatment_patients = set(sample_annotation_df[sample_annotation_df['treatment_status'] == 'pre-treatment']['patient_number'])
post_treatment_patients = set(sample_annotation_df[sample_annotation_df['treatment_status'] == 'post-treatment']['patient_number'])

# Find patients with both pre and post treatment samples
paired_patients = pre_treatment_patients.intersection(post_treatment_patients)

print(f"Total unique patients in pre-treatment: {len(pre_treatment_patients)}")
print(f"Total unique patients in post-treatment: {len(post_treatment_patients)}")
print(f"Patients with both pre and post treatment samples: {len(paired_patients)}")
print(f"Paired patients: {sorted(paired_patients)}")

# Show treatment status counts
print("\nTreatment status distribution:")
print(sample_annotation_df['treatment_status'].value_counts())

# Show detailed breakdown for paired patients
print(f"\nDetailed view of paired patients (first 10):")
paired_sample_view = sample_annotation_df[sample_annotation_df['patient_number'].isin(list(paired_patients)[:10])]
paired_sample_view.sort_values(['patient_number', 'treatment_status'])

Total unique patients in pre-treatment: 60
Total unique patients in post-treatment: 26
Patients with both pre and post treatment samples: 26
Paired patients: ['Patient01', 'Patient02', 'Patient03', 'Patient04', 'Patient05', 'Patient06', 'Patient08', 'Patient11', 'Patient12', 'Patient13', 'Patient14', 'Patient20', 'Patient22', 'Patient23', 'Patient27', 'Patient28', 'Patient29', 'Patient30', 'Patient31', 'Patient37', 'Patient38', 'Patient39', 'Patient49', 'Patient56', 'Patient59', 'Patient62']

Treatment status distribution:
treatment_status
pre-treatment     95
post-treatment    27
Name: count, dtype: int64

Detailed view of paired patients (first 10):


,sample_name,tissue,timing biopsies,patient id,Sex,age diagnostic,statut menopausal,tum size_max_diagnostic_(larger_diameter_in_mm),molecular category,ki67 binary_(1_>_and_=_15%_et_0_<15%),...,nac category,nac herceptin_binary,breast conservative_surgery,sentinel lymph_node,radiotherapy binary,hormonotherapy type_category,sequencing batch,mapped to_intergenic_(%_reads),treatment_status,patient_number
42,GSM7707304,mammary tumor,post-surgery,R03_UCL,female,34,no,40,Luminal B,1,...,anthracyclin alone,no,no,no,no,Arimidex,batch_dec_2022,22.925,post-treatment,Patient03
2,GSM7707264,mammary tumor,pre-surgery,R03_UCL,female,34,no,40,Luminal B,1,...,anthracyclin alone,no,no,no,no,Arimidex,batch_oct_2019,9.08,pre-treatment,Patient03
60,GSM7707322,mammary tumor,post-surgery,P04_UCL,female,39,no,28.6,Luminal B,1,...,anthracyclines and taxanes,no,no,no,yes,Nolvadex,batch_dec_2022,18.8440129992194,post-treatment,Patient04
3,GSM7707265,mammary tumor,pre-surgery,R04_UCL,female,75,yes,8,Luminal A,0,...,anthracyclin alone,no,no,no,yes,Femara,batch_oct_2019,7.41,pre-treatment,Patient04
69,GSM7707331,mammary tumor,pre-surgery,P04_UCL,female,39,no,28.6,Luminal B,1,...,anthracyclines and taxanes,no,no,no,yes,Nolvadex,batch_dec_2021,19.72,pre-treatment,Patient04
61,GSM7707323,mammary tumor,post-surgery,P05_UCL,female,62,yes,100,Luminal B,1,...,anthracyclines and taxanes,no,no,no,yes,Femara,batch_dec_2022,54.263,post-treatment,Patient05
4,GSM7707266,mammary tumor,pre-surgery,R05_UCL,female,32,no,29,Luminal B,1,...,anthracyclines and taxanes,no,no,no,yes,Femara,batch_oct_2019,7.85,pre-treatment,Patient05
70,GSM7707332,mammary tumor,pre-surgery,P05_UCL,female,62,yes,100,Luminal B,1,...,anthracyclines and taxanes,no,no,no,yes,Femara,batch_dec_2021,49.2,pre-treatment,Patient05
44,GSM7707306,mammary tumor,post-surgery,R08_UCL,female,44,no,45,Luminal B,1,...,anthracyclines and taxanes,no,no,no,yes,Arimidex,batch_dec_2022,8.028,post-treatment,Patient08
7,GSM7707269,mammary tumor,pre-surgery,R08_UCL,female,44,no,45,Luminal B,1,...,anthracyclines and taxanes,no,no,no,yes,Arimidex,batch_oct_2019,8.1,pre-treatment,Patient08


In [6]:
# Read bulk RNASeq raw count data
raw_counts = pd.read_csv('../data/GSE240671_raw_counts_GRCh38.p13_NCBI.tsv', sep='\t', index_col=0)
raw_counts

,GSM7707262,GSM7707266,GSM7707267,GSM7707268,GSM7707269,GSM7707270,GSM7707271,GSM7707272,GSM7707274,GSM7707275,...,GSM7707366,GSM7707367,GSM7707368,GSM7707371,GSM7707372,GSM7707373,GSM7707374,GSM7707375,GSM7707376,GSM7707377
GeneID,,,,,,,,,,,,,,,,,,,,,
100287102,7,11,15,3,6,8,2,33,6,4,...,18,11,19,33,14,6,16,22,22,20
653635,191,280,307,184,170,352,62,73,112,64,...,281,115,335,81,213,192,133,262,120,485
102466751,7,9,8,8,10,7,1,9,5,1,...,46,4,33,5,4,3,16,25,3,40
107985730,0,1,2,1,0,0,0,4,1,0,...,5,4,1,8,3,1,7,3,7,1
100302278,0,1,0,0,0,0,0,1,0,0,...,1,0,0,2,2,0,2,8,3,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4541,659,1979,1254,1821,1091,2202,729,474,1589,2723,...,3357,348,9054,384,1206,2240,1793,1319,786,5299
4556,18,57,32,57,32,40,10,58,54,91,...,196,51,430,90,66,222,99,80,62,92
4519,1942,5010,4020,3352,3542,7992,2383,735,6693,8215,...,4906,740,12067,1006,2401,10985,1883,1504,1566,11183
